# Empirical Validation of Boiling Point Model

This notebook implements a complete empirical validation of the ISA (International Standard Atmosphere) barometric formula combined with the Antoine equation for water boiling point prediction. The experiment validates against two independent empirical datasets: EngineeringToolbox and Deleze.

### Key Findings:
1. **Accuracy**: The ISA+Antoine model achieves low RMSE on empirical data, significantly outperforming an isothermal baseline.
2. **Lapse Rate Sensitivity**: Explores optimal temperature lapse rates ($L$) for different datasets.
3. **Monotonicity**: Verifies monotonic decrease of boiling point with altitude.

In [ ]:
# Install dependencies (Colab-safe guard)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.run(["pip", "install", "-q", "loguru", "numpy", "matplotlib"], check=True)
else:
    print("Running locally.")

In [ ]:
import json
import math
import sys
from pathlib import Path
from typing import List, Tuple, Dict, Any
import numpy as np
import matplotlib.pyplot as plt

# Plotting style configuration (Mandatory house style)
COLOURS = {
    "method": "#0173B2",  # proposed method / ISA+Antoine
    "baseline": "#949494",  # baseline / isothermal
    "other_1": "#DE8F05",
    "other_2": "#029E73",
    "other_3": "#CC78BC",
    "other_4": "#CA9161",
    "other_5": "#56B4E9",
}
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "#CCCCCC",
    "grid.alpha": 0.6,
    "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=list(COLOURS.values())),
    "image.cmap": "cividis",
    "legend.frameon": False,
    "figure.constrained_layout.use": True,
    "figure.dpi": 120,
})

In [ ]:
# Data loading helper with GitHub URL and local fallback
import http.client
import urllib.request

GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-daa948-boiling-point-prediction-error-as/main/round-2/experiment-3/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=15) as response:
            return json.loads(response.read().decode())
    except (OSError, ValueError, http.client.HTTPException) as e:
        print("GitHub data unavailable, using local mini_demo_data.json:", e)
    local = Path("mini_demo_data.json")
    if local.exists():
        return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
print("Loaded datasets keys:", list(data.keys()))

### Model Constants and Core Equations

We define the ISA barometric pressure formula, the Antoine vapor pressure equation for water, and the isothermal baseline model.

In [ ]:
# Physical constants
P0 = 1013.25  # hPa, sea level pressure
T0 = 288.15   # K, sea level temperature
g = 9.80665   # m/s^2
M = 0.0289644 # kg/mol
R = 8.31447   # J/(mol*K)
L = 0.0065    # K/m, standard lapse rate

# Antoine equation constants for water
A = 8.07131
B = 1730.63
C = 233.426

def isa_pressure(altitude_m: float, lapse_rate: float = L) -> float:
    if lapse_rate == 0:
        return P0 * math.exp(-g * M * altitude_m / (R * T0))
    exponent = g * M / (R * lapse_rate)
    term = 1 - lapse_rate * altitude_m / T0
    if term <= 0:
        return 0.0
    return P0 * (term ** exponent)

def antoine_boiling_point(pressure_hpa: float) -> float:
    if pressure_hpa <= 0:
        return -273.15
    pressure_mmhg = pressure_hpa * 0.750062
    return B / (A - math.log10(pressure_mmhg)) - C

def boiling_point_at_altitude(altitude_m: float, lapse_rate: float = L) -> float:
    pressure = isa_pressure(altitude_m, lapse_rate)
    return antoine_boiling_point(pressure)

def baseline_pressure(altitude_m: float) -> float:
    return P0 * math.exp(-g * M * altitude_m / (R * T0))

def baseline_boiling_point(altitude_m: float) -> float:
    pressure = baseline_pressure(altitude_m)
    return antoine_boiling_point(pressure)

### Reference Points & Monotonicity Check

In [ ]:
# Reference point tests
reference_points = [
    (0, 100.0),
    (1000, 96.7),
    (2000, 93.5),
    (3000, 90.4),
    (5000, 84.4)
]

print("Reference Point Validation:")
for alt, expected_bp in reference_points:
    calc_bp = boiling_point_at_altitude(alt)
    print(f"  Altitude {alt:4d}m: Expected {expected_bp:.1f}°C, Calculated {calc_bp:.2f}°C, Diff {expected_bp - calc_bp:+.2f}°C")

# Verify monotonic decrease
test_alts = [0, 1000, 2000, 3000, 5000]
bps = [boiling_point_at_altitude(a) for a in test_alts]
is_decreasing = all(bps[i] > bps[i+1] for i in range(len(bps)-1))
print(f"\nMonotonically decreasing with altitude: {is_decreasing}")

### Empirical Validation and Sensitivity Analysis

In [ ]:
def compute_metrics(observed: List[float], predicted: List[float]) -> Dict[str, float]:
    obs = np.array(observed)
    pred = np.array(predicted)
    mse = np.mean((obs - pred) ** 2)
    rmse = math.sqrt(mse)
    bias = np.mean(obs - pred)
    return {
        "rmse": float(rmse),
        "bias": float(bias),
        "mae": float(np.mean(np.abs(obs - pred))),
        "n_samples": len(obs)
    }

def run_validation(empirical_list, lapse_rate=L, model_name="ISA+Antoine"):
    altitudes = [d["altitude_m"] for d in empirical_list]
    observed = [d["boiling_point_c"] for d in empirical_list]
    
    if model_name == "ISA+Antoine":
        predicted = [boiling_point_at_altitude(alt, lapse_rate) for alt in altitudes]
    elif model_name == "Baseline":
        predicted = [baseline_boiling_point(alt) for alt in altitudes]
    else:
        raise ValueError(f"Unknown model: {model_name}")
    
    metrics = compute_metrics(observed, predicted)
    return {"model": model_name, "metrics": metrics, "altitudes": altitudes, "observed": observed, "predicted": predicted}

# Run on loaded data
et_data = data["engineering_toolbox"]
del_data = data["deleze"]

res_et_isa = run_validation(et_data, L, "ISA+Antoine")
res_et_base = run_validation(et_data, L, "Baseline")

res_del_isa = run_validation(del_data, L, "ISA+Antoine")
res_del_base = run_validation(del_data, L, "Baseline")

print("EngineeringToolbox Validation:")
print(f"  ISA+Antoine: RMSE = {res_et_isa[metrics][rmse]:.4f}°C, Bias = {res_et_isa[metrics][bias]:.4f}°C")
print(f"  Baseline:    RMSE = {res_et_base[metrics][rmse]:.4f}°C, Bias = {res_et_base[metrics][bias]:.4f}°C")

print("Deleze Validation:")
print(f"  ISA+Antoine: RMSE = {res_del_isa[metrics][rmse]:.4f}°C, Bias = {res_del_isa[metrics][bias]:.4f}°C")
print(f"  Baseline:    RMSE = {res_del_base[metrics][rmse]:.4f}°C, Bias = {res_del_base[metrics][bias]:.4f}°C")

### Results Visualization & Summary

In [ ]:
# Plotting validation results
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Boiling Point vs Altitude (EngineeringToolbox)
axes[0].scatter(res_et_isa["altitudes"], res_et_isa["observed"], color=COLOURS["baseline"], label="Observed (EngToolbox)", s=25, alpha=0.7)
axes[0].plot(res_et_isa["altitudes"], res_et_isa["predicted"], color=COLOURS["method"], label="ISA+Antoine", linewidth=2)
axes[0].set_title("Water Boiling Point vs Altitude")
axes[0].set_xlabel("Altitude (m)")
axes[0].set_ylabel("Boiling Point (°C)")
axes[0].legend()

# Plot 2: Error vs Altitude
errors_isa = np.array(res_et_isa["observed"]) - np.array(res_et_isa["predicted"])
errors_base = np.array(res_et_base["observed"]) - np.array(res_et_base["predicted"])

axes[1].plot(res_et_isa["altitudes"], errors_isa, color=COLOURS["method"], label="ISA+Antoine Error", linewidth=1.5)
axes[1].plot(res_et_base["altitudes"], errors_base, color=COLOURS["baseline"], label="Baseline Error", linewidth=1.5, linestyle="--")
axes[1].axhline(0, color="black", linewidth=0.8, linestyle=":")
axes[1].set_title("Prediction Residuals (Observed - Predicted)")
axes[1].set_xlabel("Altitude (m)")
axes[1].set_ylabel("Error (°C)")
axes[1].legend()

plt.show()

print("\nDemo execution completed successfully!")